# Thoracolumbar Coverage Strategy Clean - Colab

Version para Google Colab del notebook de analisis de cobertura.

Este notebook asume que todo el proyecto esta dentro de una carpeta de
Google Drive y que mantuviste la misma estructura de archivos.

Fuentes principales:
- `indice_dataset.csv`
- `diccionario_etiquetas_T1_T12_L1_L5.json`
- `reporte_por_mascara_version_final.csv`
- `resumen_version_final_T1_T12_L1_L5.csv`

Salidas esperadas:
- `analysis_outputs/thoracolumbar_coverage_summary.csv`
- `analysis_outputs/thoracolumbar_presence_matrix.csv`
- `analysis_outputs/training_manifest_thoracolumbar.csv`
- `analysis_outputs/thoracolumbar_review_candidates.csv`

## 0. Preparacion de Colab

Antes de correr el resto:

1. sube tu carpeta del proyecto a Google Drive
2. conserva la estructura interna de carpetas
3. ajusta `PROJECT_ROOT` en la siguiente celda

In [9]:
from google.colab import drive
drive.mount('/content/drive')

import os
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DataRadriografias")
if not PROJECT_ROOT.exists():
    raise FileNotFoundError(f"No existe la carpeta: {PROJECT_ROOT}")

os.chdir(PROJECT_ROOT)
print("Working directory:", Path.cwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/DataRadriografias


## 1. Carga inicial

In [10]:
from __future__ import annotations

import json
import shutil
import time
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
DATASET_ROOT = ROOT / 'data' / 'Scoliosis_Dataset'
LOCAL_CACHE_DIR = Path('/tmp/data_radiografias_colab_cache')
LOCAL_CACHE_DIR.mkdir(parents=True, exist_ok=True)
ANALYSIS_DIR = ROOT / 'analysis_outputs'
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

INDEX_PATH = ROOT / 'indice_dataset.csv'
DICT_PATH = ROOT / 'diccionario_etiquetas_T1_T12_L1_L5.json'
REPORT_PATH = ROOT / 'reporte_por_mascara_version_final.csv'
SUMMARY_FINAL_PATH = ROOT / 'resumen_version_final_T1_T12_L1_L5.csv'
SUMMARY_ORIGINAL_PATH = ROOT / 'resumen_diccionario_original_35_entidades.csv'

assert INDEX_PATH.exists(), 'No se encontro indice_dataset.csv'
assert DICT_PATH.exists(), 'No se encontro diccionario_etiquetas_T1_T12_L1_L5.json'
assert REPORT_PATH.exists(), 'No se encontro reporte_por_mascara_version_final.csv'
assert SUMMARY_FINAL_PATH.exists(), 'No se encontro resumen_version_final_T1_T12_L1_L5.csv'

index_df_raw = pd.read_csv(INDEX_PATH)
report_df_raw = pd.read_csv(REPORT_PATH)
summary_final_df = pd.read_csv(SUMMARY_FINAL_PATH)
summary_original_df = pd.read_csv(SUMMARY_ORIGINAL_PATH) if SUMMARY_ORIGINAL_PATH.exists() else pd.DataFrame()

with open(DICT_PATH, 'r', encoding='utf-8') as f:
    labels_dict = json.load(f)

print('ROOT:', ROOT)
print('Filas indice:', len(index_df_raw))
print('Filas reporte:', len(report_df_raw))
print('Filas resumen final:', len(summary_final_df))


def resolve_dataset_path(rel: str) -> str:
    rel_path = Path(str(rel).replace(chr(92), '/'))
    if rel_path.is_absolute():
        return str(rel_path)
    candidates = [
        ROOT / rel_path,
        DATASET_ROOT / rel_path,
    ]
    for candidate in candidates:
        if candidate.exists():
            return str(candidate.resolve())
    return str((DATASET_ROOT / rel_path).resolve())


def safe_copy_to_drive(local_path: Path, destination_path: Path, retries: int = 5, sleep_seconds: float = 2.0) -> None:
    destination_path.parent.mkdir(parents=True, exist_ok=True)
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            shutil.copy2(local_path, destination_path)
            return
        except Exception as exc:
            last_error = exc
            print(f'Intento {attempt}/{retries} fallido copiando a Drive: {destination_path} -> {exc}')
            time.sleep(sleep_seconds)
    raise last_error


def safe_to_csv(df: pd.DataFrame, destination_path: Path, index: bool = False) -> None:
    local_path = LOCAL_CACHE_DIR / destination_path.name
    df.to_csv(local_path, index=index)
    safe_copy_to_drive(local_path, destination_path)

ROOT: /content/drive/MyDrive/DataRadriografias
Filas indice: 250
Filas reporte: 250
Filas resumen final: 18


## 2. Normalizacion de columnas

In [11]:
index_col_map = {
    'grupo': 'split',
    'imagen': 'image',
    'id_paciente': 'patient_id',
    'ruta_radiografia': 'radiograph_path',
    'ruta_mascara_binaria': 'label_binary_path',
    'ruta_mascara_multiclase_id_png': 'multiclass_id_png',
    'ruta_mascara_multiclase_grises_jpg': 'multiclass_gray_jpg',
    'ruta_mascara_multiclase_color_jpg': 'multiclass_color_jpg',
    'metricas_json': 'metrics_json',
}

report_col_map = {
    'archivo_mascara': 'mask_filename',
    'etiquetas_presentes_en_version_final': 'present_final_labels',
    'otras_etiquetas_identificadas_en_mascara_original': 'other_original_labels',
    'numero_otras_etiquetas_identificadas': 'other_original_labels_count',
    'numero_etiquetas_presentes_en_version_final': 'num_present_final_labels',
}

index_df = index_df_raw.rename(columns=index_col_map).copy()
report_df = report_df_raw.rename(columns=report_col_map).copy()

required_index_cols = ['split', 'image', 'patient_id', 'radiograph_path', 'label_binary_path', 'multiclass_id_png']
missing_index_cols = [col for col in required_index_cols if col not in index_df.columns]
if missing_index_cols:
    raise ValueError(f'Faltan columnas esperadas en el indice: {missing_index_cols}')

display(index_df.head())
display(report_df.head())
display(summary_final_df)

,split,image,patient_id,radiograph_path,label_binary_path,multiclass_id_png,multiclass_gray_jpg,multiclass_color_jpg,metrics_json
0,Normal,N_1.jpg,1,Normal/N_1.jpg,LabelBinaryJPG/Label_N_1.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_1.png,LabelMultiClass_Gray_JPG/LabelMulti_N_1.jpg,LabelMultiClass_Color_JPG/LabelMulti_N_1.jpg,NaN
1,Normal,N_2.jpg,2,Normal/N_2.jpg,LabelBinaryJPG/Label_N_2.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_2.png,LabelMultiClass_Gray_JPG/LabelMulti_N_2.jpg,LabelMultiClass_Color_JPG/LabelMulti_N_2.jpg,NaN
2,Normal,N_3.jpg,3,Normal/N_3.jpg,LabelBinaryJPG/Label_N_3.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_3.png,LabelMultiClass_Gray_JPG/LabelMulti_N_3.jpg,LabelMultiClass_Color_JPG/LabelMulti_N_3.jpg,NaN
3,Normal,N_4.jpg,4,Normal/N_4.jpg,LabelBinaryJPG/Label_N_4.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_4.png,LabelMultiClass_Gray_JPG/LabelMulti_N_4.jpg,LabelMultiClass_Color_JPG/LabelMulti_N_4.jpg,NaN
4,Normal,N_5.jpg,5,Normal/N_5.jpg,LabelBinaryJPG/Label_N_5.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_5.png,LabelMultiClass_Gray_JPG/LabelMulti_N_5.jpg,LabelMultiClass_Color_JPG/LabelMulti_N_5.jpg,NaN


,mask_filename,present_final_labels,other_original_labels,other_original_labels_count,num_present_final_labels
0,LabelMulti_N_1.png,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","C7, C6, C5, C4",4,17
1,LabelMulti_N_10.png,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","C7, C6",2,17
2,LabelMulti_N_11.png,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","C7, C6, C5",3,17
3,LabelMulti_N_12.png,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","C7, C6",2,17
4,LabelMulti_N_13.png,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...","C7, C6",2,17


,id_final,vertebra,numero_imagenes,total_pixeles
0,0,background,250,249208878
1,1,T1,238,792441
2,2,T2,245,1264592
3,3,T3,247,1172832
4,4,T4,246,1181918
5,5,T5,241,1240650
6,6,T6,235,1206119
7,7,T7,231,1296989
8,8,T8,225,1363220
9,9,T9,221,1426601


## 3. Target final

In [12]:
final_multiclass_map = {int(k): v for k, v in labels_dict['mascara_multiclase_id_png'].items()}

target_labels = [f'T{i}' for i in range(1, 13)] + [f'L{i}' for i in range(1, 6)]
thoracic_labels = [f'T{i}' for i in range(1, 13)]
lumbar_labels = [f'L{i}' for i in range(1, 6)]
historical_cervical_labels = ['C7', 'C6', 'C5', 'C4', 'C3']

label_to_final_id = {label: idx for idx, label in final_multiclass_map.items()}
missing_target_labels = [label for label in target_labels if label not in label_to_final_id]
if missing_target_labels:
    raise ValueError(f'Faltan etiquetas target en el diccionario final: {missing_target_labels}')

display(pd.DataFrame({'target_labels': target_labels}))

,target_labels
0,T1
1,T2
2,T3
3,T4
4,T5
5,T6
6,T7
7,T8
8,T9
9,T10


## 4. Cobertura por imagen

In [13]:
def split_label_string(value: object) -> list[str]:
    if pd.isna(value):
        return []
    text = str(value).strip()
    if text == '':
        return []
    return [item.strip() for item in text.split(',') if item.strip()]


def ordered_visible_labels(labels_present: list[str], canonical_order: list[str]) -> list[str]:
    present_set = set(labels_present)
    return [label for label in canonical_order if label in present_set]


def internal_missing_labels(visible_labels: list[str], canonical_order: list[str]) -> list[str]:
    if not visible_labels:
        return []
    positions = [canonical_order.index(label) for label in visible_labels]
    first_pos = min(positions)
    last_pos = max(positions)
    span_labels = canonical_order[first_pos:last_pos + 1]
    visible_set = set(visible_labels)
    return [label for label in span_labels if label not in visible_set]


report_df['mask_filename'] = report_df['mask_filename'].astype(str).str.strip()
report_df['present_final_labels'] = report_df['present_final_labels'].fillna('')
report_df['other_original_labels'] = report_df['other_original_labels'].fillna('')
report_df['other_original_labels_count'] = pd.to_numeric(report_df['other_original_labels_count'], errors='coerce').fillna(0).astype(int)
report_df['num_present_final_labels'] = pd.to_numeric(report_df['num_present_final_labels'], errors='coerce').fillna(0).astype(int)

index_df['mask_filename'] = index_df['multiclass_id_png'].apply(lambda p: Path(str(p)).name.strip())

merged_df = index_df.merge(report_df, on='mask_filename', how='left', indicator=True)
missing_mask_rows = merged_df.loc[merged_df['_merge'] != 'both', ['split', 'image', 'mask_filename']]
if not missing_mask_rows.empty:
    raise ValueError(f'Hay mascaras sin cruce en el reporte por mascara: {missing_mask_rows.head(10).to_dict(orient="records")}')
merged_df = merged_df.drop(columns=['_merge'])

coverage_rows = []
for _, row in merged_df.iterrows():
    present_final_labels = split_label_string(row['present_final_labels'])
    ordered_target = ordered_visible_labels(present_final_labels, target_labels)
    ordered_thoracic = ordered_visible_labels(present_final_labels, thoracic_labels)
    ordered_lumbar = ordered_visible_labels(present_final_labels, lumbar_labels)

    thoracic_missing = internal_missing_labels(ordered_thoracic, thoracic_labels)
    lumbar_missing = internal_missing_labels(ordered_lumbar, lumbar_labels)

    other_original_labels = split_label_string(row['other_original_labels'])
    other_original_cervical = [label for label in other_original_labels if label in historical_cervical_labels]
    other_original_non_cervical = [label for label in other_original_labels if label not in historical_cervical_labels]

    coverage_rows.append(
        {
            'split': row['split'],
            'image': row['image'],
            'patient_id': row['patient_id'],
            'radiograph_path': row['radiograph_path'],
            'mask_path': row['multiclass_id_png'],
            'num_visible_target_vertebrae': len(ordered_target),
            'first_visible_target': ordered_target[0] if ordered_target else '',
            'last_visible_target': ordered_target[-1] if ordered_target else '',
            'visible_target_span_signature': f'{ordered_target[0]}-{ordered_target[-1]}' if ordered_target else 'no_target_vertebra',
            'present_target_vertebrae': ', '.join(ordered_target),
            'historical_cervical_trace_labels': ', '.join(other_original_cervical),
            'historical_cervical_trace_count': len(other_original_cervical),
            'other_original_non_cervical_labels': ', '.join(other_original_non_cervical),
            'other_original_non_cervical_count': len(other_original_non_cervical),
            'thoracic_count': len(ordered_thoracic),
            'lumbar_count': len(ordered_lumbar),
            'thoracic_internal_missing_count': len(thoracic_missing),
            'lumbar_internal_missing_count': len(lumbar_missing),
            'total_internal_missing_count': len(thoracic_missing) + len(lumbar_missing),
            'thoracic_internal_missing_labels': ', '.join(thoracic_missing),
            'lumbar_internal_missing_labels': ', '.join(lumbar_missing),
            'has_thoracic': len(ordered_thoracic) > 0,
            'has_lumbar': len(ordered_lumbar) > 0,
            'has_full_target_range_t1_l5': ordered_target == target_labels,
        }
    )

coverage_df = pd.DataFrame(coverage_rows)
coverage_df['radiograph_path_abs'] = coverage_df['radiograph_path'].apply(resolve_dataset_path)
coverage_df['mask_path_abs'] = coverage_df['mask_path'].apply(resolve_dataset_path)

display(coverage_df.head())

,split,image,patient_id,radiograph_path,mask_path,num_visible_target_vertebrae,first_visible_target,last_visible_target,visible_target_span_signature,present_target_vertebrae,...,thoracic_internal_missing_count,lumbar_internal_missing_count,total_internal_missing_count,thoracic_internal_missing_labels,lumbar_internal_missing_labels,has_thoracic,has_lumbar,has_full_target_range_t1_l5,radiograph_path_abs,mask_path_abs
0,Normal,N_1.jpg,1,Normal/N_1.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_1.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
1,Normal,N_2.jpg,2,Normal/N_2.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_2.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
2,Normal,N_3.jpg,3,Normal/N_3.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_3.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
3,Normal,N_4.jpg,4,Normal/N_4.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_4.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
4,Normal,N_5.jpg,5,Normal/N_5.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_5.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...


## 5. Matriz de presencia y resumen

In [14]:
presence_rows = []
for _, row in coverage_df.iterrows():
    present_labels = split_label_string(row['present_target_vertebrae'])
    record = {'split': row['split'], 'image': row['image'], 'patient_id': row['patient_id']}
    for label in target_labels:
        record[f'present_{label}'] = int(label in present_labels)
    presence_rows.append(record)

presence_df = pd.DataFrame(presence_rows)

vertebra_coverage_rows = []
for label in target_labels:
    count = int(presence_df[f'present_{label}'].sum())
    vertebra_coverage_rows.append(
        {
            'vertebra_label': label,
            'images_with_label': count,
            'coverage_rate': count / len(presence_df),
            'region': 'thoracic' if label.startswith('T') else 'lumbar',
        }
    )

vertebra_coverage_df = pd.DataFrame(vertebra_coverage_rows)

summary_df = pd.DataFrame(
    [
        {'metric': 'total_images', 'value': len(coverage_df)},
        {'metric': 'images_with_any_target_vertebra', 'value': int((coverage_df['num_visible_target_vertebrae'] > 0).sum())},
        {'metric': 'images_with_thoracic_region', 'value': int(coverage_df['has_thoracic'].sum())},
        {'metric': 'images_with_lumbar_region', 'value': int(coverage_df['has_lumbar'].sum())},
        {'metric': 'images_with_full_target_range_t1_l5', 'value': int(coverage_df['has_full_target_range_t1_l5'].sum())},
        {'metric': 'images_with_internal_missing', 'value': int((coverage_df['total_internal_missing_count'] > 0).sum())},
        {'metric': 'images_with_historical_cervical_trace', 'value': int((coverage_df['historical_cervical_trace_count'] > 0).sum())},
        {'metric': 'images_with_other_original_non_cervical_labels', 'value': int((coverage_df['other_original_non_cervical_count'] > 0).sum())},
        {'metric': 'mean_visible_target_vertebrae', 'value': float(coverage_df['num_visible_target_vertebrae'].mean())},
        {'metric': 'median_visible_target_vertebrae', 'value': float(coverage_df['num_visible_target_vertebrae'].median())},
    ]
)

display(vertebra_coverage_df)
display(summary_df)

,vertebra_label,images_with_label,coverage_rate,region
0,T1,238,0.952,thoracic
1,T2,245,0.980,thoracic
2,T3,247,0.988,thoracic
3,T4,246,0.984,thoracic
4,T5,241,0.964,thoracic
5,T6,235,0.940,thoracic
6,T7,231,0.924,thoracic
7,T8,225,0.900,thoracic
8,T9,221,0.884,thoracic
9,T10,215,0.860,thoracic


,metric,value
0,total_images,250.000
1,images_with_any_target_vertebra,249.000
2,images_with_thoracic_region,249.000
3,images_with_lumbar_region,164.000
4,images_with_full_target_range_t1_l5,73.000
5,images_with_internal_missing,8.000
6,images_with_historical_cervical_trace,213.000
7,images_with_other_original_non_cervical_labels,7.000
8,mean_visible_target_vertebrae,13.524
9,median_visible_target_vertebrae,15.000


## 6. Casos de revision y manifest

In [15]:
review_candidates_df = coverage_df.loc[
    coverage_df['total_internal_missing_count'].gt(0)
    | coverage_df['other_original_non_cervical_count'].gt(0)
    | coverage_df['num_visible_target_vertebrae'].eq(0)
].copy()

CORE_THRESHOLD = 0.80
MIN_VISIBLE_TARGET_VERTEBRAE = 6

core_labels = vertebra_coverage_df.loc[
    vertebra_coverage_df['coverage_rate'].ge(CORE_THRESHOLD),
    'vertebra_label',
].tolist()

manifest_df = coverage_df.copy()
manifest_df['unique_sample_id'] = manifest_df['split'].astype(str) + '__' + manifest_df['image'].astype(str)
manifest_df['group_id_for_split'] = manifest_df['split'].astype(str) + '_' + manifest_df['patient_id'].astype(str)
manifest_df['target_class'] = manifest_df['split'].map({'Normal': 0, 'Scoliosis': 1}).fillna(-1).astype(int)

for label in target_labels:
    manifest_df[f'present_{label}'] = presence_df[f'present_{label}']

manifest_df['all_core_labels_present'] = True
for label in core_labels:
    manifest_df['all_core_labels_present'] &= manifest_df[f'present_{label}'].eq(1)

manifest_df['usable_for_binary_spine'] = True
manifest_df['usable_for_thoracolumbar_core'] = (
    manifest_df['all_core_labels_present']
    & manifest_df['total_internal_missing_count'].eq(0)
    & manifest_df['other_original_non_cervical_count'].eq(0)
)
manifest_df['usable_for_thoracolumbar_partial'] = (
    manifest_df['num_visible_target_vertebrae'].ge(MIN_VISIBLE_TARGET_VERTEBRAE)
    & manifest_df['total_internal_missing_count'].eq(0)
    & manifest_df['other_original_non_cervical_count'].eq(0)
)
manifest_df['needs_annotation_review'] = (
    manifest_df['total_internal_missing_count'].gt(0)
    | manifest_df['other_original_non_cervical_count'].gt(0)
)
manifest_df['usable_for_cobb_regression'] = manifest_df['split'].eq('Scoliosis')

display(pd.DataFrame({'core_labels': core_labels}))
display(review_candidates_df.head(25))
display(manifest_df.head())

,core_labels
0,T1
1,T2
2,T3
3,T4
4,T5
5,T6
6,T7
7,T8
8,T9
9,T10


,split,image,patient_id,radiograph_path,mask_path,num_visible_target_vertebrae,first_visible_target,last_visible_target,visible_target_span_signature,present_target_vertebrae,...,thoracic_internal_missing_count,lumbar_internal_missing_count,total_internal_missing_count,thoracic_internal_missing_labels,lumbar_internal_missing_labels,has_thoracic,has_lumbar,has_full_target_range_t1_l5,radiograph_path_abs,mask_path_abs
13,Normal,N_14.jpg,14,Normal/N_14.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_14.png,16,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T9, T10, T11, T12,...",...,1,0,1,T8,,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
37,Normal,N_38.jpg,38,Normal/N_38.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_38.png,16,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,1,1,,L4,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
40,Normal,N_41.jpg,41,Normal/N_41.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_41.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
42,Normal,N_43.jpg,43,Normal/N_43.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_43.png,16,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T9, T10, T11, T12,...",...,1,0,1,T8,,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
47,Normal,N_48.jpg,48,Normal/N_48.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_48.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
51,Normal,N_52.jpg,52,Normal/N_52.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_52.png,16,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T11, T12, ...",...,1,0,1,T10,,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
53,Normal,N_54.jpg,54,Normal/N_54.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_54.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
58,Normal,N_59.jpg,59,Normal/N_59.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_59.png,16,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,1,1,,L2,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
60,Normal,N_61.jpg,61,Normal/N_61.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_61.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,0,0,0,,,True,True,True,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...
64,Normal,N_65.jpg,65,Normal/N_65.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_65.png,15,T1,L4,T1-L4,"T1, T2, T3, T4, T5, T6, T8, T9, T10, T11, T12,...",...,1,0,1,T7,,True,True,False,/content/drive/MyDrive/DataRadriografias/data/...,/content/drive/MyDrive/DataRadriografias/data/...


,split,image,patient_id,radiograph_path,mask_path,num_visible_target_vertebrae,first_visible_target,last_visible_target,visible_target_span_signature,present_target_vertebrae,...,present_L2,present_L3,present_L4,present_L5,all_core_labels_present,usable_for_binary_spine,usable_for_thoracolumbar_core,usable_for_thoracolumbar_partial,needs_annotation_review,usable_for_cobb_regression
0,Normal,N_1.jpg,1,Normal/N_1.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_1.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,1,1,1,1,True,True,True,True,False,False
1,Normal,N_2.jpg,2,Normal/N_2.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_2.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,1,1,1,1,True,True,True,True,False,False
2,Normal,N_3.jpg,3,Normal/N_3.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_3.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,1,1,1,1,True,True,True,True,False,False
3,Normal,N_4.jpg,4,Normal/N_4.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_4.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,1,1,1,1,True,True,True,True,False,False
4,Normal,N_5.jpg,5,Normal/N_5.jpg,LabelMultiClass_ID_PNG/LabelMulti_N_5.png,17,T1,L5,T1-L5,"T1, T2, T3, T4, T5, T6, T7, T8, T9, T10, T11, ...",...,1,1,1,1,True,True,True,True,False,False


## 7. Exportacion

In [16]:
coverage_output_path = ANALYSIS_DIR / 'thoracolumbar_coverage_summary.csv'
presence_output_path = ANALYSIS_DIR / 'thoracolumbar_presence_matrix.csv'
manifest_output_path = ANALYSIS_DIR / 'training_manifest_thoracolumbar.csv'
review_output_path = ANALYSIS_DIR / 'thoracolumbar_review_candidates.csv'

safe_to_csv(coverage_df, coverage_output_path, index=False)
safe_to_csv(presence_df, presence_output_path, index=False)
safe_to_csv(manifest_df, manifest_output_path, index=False)
safe_to_csv(review_candidates_df, review_output_path, index=False)

print('Archivo guardado:', coverage_output_path)
print('Archivo guardado:', presence_output_path)
print('Archivo guardado:', manifest_output_path)
print('Archivo guardado:', review_output_path)

Archivo guardado: /content/drive/MyDrive/DataRadriografias/analysis_outputs/thoracolumbar_coverage_summary.csv
Archivo guardado: /content/drive/MyDrive/DataRadriografias/analysis_outputs/thoracolumbar_presence_matrix.csv
Archivo guardado: /content/drive/MyDrive/DataRadriografias/analysis_outputs/training_manifest_thoracolumbar.csv
Archivo guardado: /content/drive/MyDrive/DataRadriografias/analysis_outputs/thoracolumbar_review_candidates.csv
